In [1]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import json

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR    = Path("../models")

PLOT_CFG = dict(
    width=1200, height=700,
    margin=dict(l=50, r=50, t=50, b=120),
    legend=dict(
        orientation="h", yanchor="top",
        y=-0.15, xanchor="center", x=0.5,
        font=dict(size=10)
    )
)

# ── load ───────────────────────────────────────────────────────────
df = pl.read_parquet(PROCESSED_DIR / "legs_fleet_daily.parquet")

# ── same split boundaries as availability ─────────────────────────
TRAIN_END  = "2025-09-30"
VAL_START  = "2025-10-01"
VAL_END    = "2025-12-31"

df_train = df.filter(pl.col("dep_date") <= pl.lit(TRAIN_END).str.to_date())
df_val   = df.filter(
    (pl.col("dep_date") >= pl.lit(VAL_START).str.to_date()) &
    (pl.col("dep_date") <= pl.lit(VAL_END).str.to_date())
)

print(f"Train: {df_train['dep_date'].min()} → {df_train['dep_date'].max()} ({len(df_train)} days)")
print(f"Val:   {df_val['dep_date'].min()} → {df_val['dep_date'].max()} ({len(df_val)} days)")

# ── load cancellation lookup ───────────────────────────────────────
with open(PROCESSED_DIR / "cancellation_lookup.json") as f:
    cancel_lookup = json.load(f)
print(f"\nCancellation lookup: {cancel_lookup}")

Train: 2024-01-01 → 2025-09-30 (639 days)
Val:   2025-10-01 → 2025-12-31 (92 days)

Cancellation lookup: {'0-7d': 0.1714, '31-90d': 0.2327, '8-30d': 0.1783, '91d+': 0.3321}


In [2]:
TARGET_COLS = ["program_live_legs", "non_program_live_legs"]

# monthly mean from training data
monthly_means_legs = (
    df_train
    .with_columns(pl.col("dep_date").dt.month().alias("month"))
    .group_by("month")
    .agg([pl.col(c).mean().alias(f"{c}_baseline") for c in TARGET_COLS])
    .sort("month")
)

# apply to validation
df_val_bl = (
    df_val
    .with_columns(pl.col("dep_date").dt.month().alias("month"))
    .join(monthly_means_legs, on="month", how="left")
)

results_baseline = {}
for col in TARGET_COLS:
    actual    = df_val_bl[col].to_numpy()
    predicted = df_val_bl[f"{col}_baseline"].to_numpy()
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    mae  = np.mean(np.abs(actual - predicted))
    results_baseline[col] = {"RMSE": round(rmse, 4), "MAE": round(mae, 4)}
    print(f"{col:30s} → RMSE: {rmse:.4f}  MAE: {mae:.4f}")

program_live_legs              → RMSE: 7.5173  MAE: 5.8031
non_program_live_legs          → RMSE: 2.8244  MAE: 2.2097


In [3]:
def fit_evaluate_prophet_legs(
    df_train, df_val, target_col, monthly_means_legs, results_baseline
):
    """Prophet model for leg count targets (program + non-program)."""

    # ── ADF test ─────────────────────────────────────────────────
    series = df_train[target_col].to_numpy()
    adf_p  = adfuller(series, autolag="AIC")[1]
    print(f"\n── {target_col} ──")
    print(f"  ADF p-value: {adf_p:.4f} → "
          f"{'STATIONARY ✅' if adf_p < 0.05 else 'NON-STATIONARY ⚠️'}")

    # ── prepare Prophet df ────────────────────────────────────────
    df_prophet = (
        df_train
        .select(["dep_date", target_col])
        .rename({"dep_date": "ds", target_col: "y"})
        .to_pandas()
    )

    # ── fit ───────────────────────────────────────────────────────
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,   # ← KEY difference from availability
        daily_seasonality=False,
        interval_width=0.80,
        changepoint_prior_scale=0.1  # slightly more flexible for count data
    )
    model.fit(df_prophet)

    # ── forecast val + Jan-May 2026 ───────────────────────────────
    future   = model.make_future_dataframe(periods=len(df_val) + 151, freq="D")
    forecast = model.predict(future)

    # ── slice validation ──────────────────────────────────────────
    val_dates = df_val["dep_date"].to_pandas()
    fc_val    = forecast[forecast["ds"].isin(val_dates)].reset_index(drop=True)
    predicted = np.clip(fc_val["yhat"].values, 0, None)  # counts ≥ 0

    # ── evaluate ──────────────────────────────────────────────────
    actual        = df_val[target_col].to_numpy()
    rmse_prophet  = np.sqrt(np.mean((actual - predicted)**2))
    mae_prophet   = np.mean(np.abs(actual - predicted))
    rmse_baseline = results_baseline[target_col]["RMSE"]
    improvement   = (rmse_baseline - rmse_prophet) / rmse_baseline * 100

    print(f"  {'':25s}  {'RMSE':>8}  {'MAE':>8}")
    print(f"  {'Baseline':25s}  {rmse_baseline:>8.4f}  {results_baseline[target_col]['MAE']:>8.4f}")
    print(f"  {'Prophet (weekly+yearly)':25s}  {rmse_prophet:>8.4f}  {mae_prophet:>8.4f}")
    print(f"  RMSE improvement: {improvement:+.1f}%")

    return model, forecast, fc_val, predicted, improvement


# ── run both models ───────────────────────────────────────────────
model_prog, fc_prog_full, fc_prog_val, prog_pred, prog_improv = \
    fit_evaluate_prophet_legs(df_train, df_val, "program_live_legs",
                              monthly_means_legs, results_baseline)

model_nonprog, fc_nonprog_full, fc_nonprog_val, nonprog_pred, nonprog_improv = \
    fit_evaluate_prophet_legs(df_train, df_val, "non_program_live_legs",
                              monthly_means_legs, results_baseline)


── program_live_legs ──
  ADF p-value: 0.0230 → STATIONARY ✅


15:32:43 - cmdstanpy - INFO - Chain [1] start processing
15:32:43 - cmdstanpy - INFO - Chain [1] done processing
15:32:43 - cmdstanpy - INFO - Chain [1] start processing
15:32:43 - cmdstanpy - INFO - Chain [1] done processing


                                 RMSE       MAE
  Baseline                     7.5173    5.8031
  Prophet (weekly+yearly)      7.0446    5.2632
  RMSE improvement: +6.3%

── non_program_live_legs ──
  ADF p-value: 0.0000 → STATIONARY ✅
                                 RMSE       MAE
  Baseline                     2.8244    2.2097
  Prophet (weekly+yearly)      2.8464    2.2396
  RMSE improvement: -0.8%


In [4]:
val_dates = df_val["dep_date"].to_list()

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Program Live Legs — Prophet vs Actual",
    f"Non-Program Live Legs — Prophet vs Actual"
])

plot_data = [
    ("program_live_legs",     prog_pred,    fc_prog_val,    "#2563eb"),
    ("non_program_live_legs", nonprog_pred, fc_nonprog_val, "#d97706"),
]

for idx, (col, pred, fc_val_df, color) in enumerate(plot_data):
    actual = df_val[col].to_list()

    # actuals
    fig.add_trace(go.Scatter(
        x=val_dates, y=actual,
        mode="lines", name=f"{col} actual",
        line=dict(color=color, width=1, dash="dot"), opacity=0.6
    ), row=1, col=idx+1)

    # prophet forecast
    fig.add_trace(go.Scatter(
        x=val_dates, y=pred.tolist(),
        mode="lines", name="Prophet forecast",
        line=dict(color=color, width=2)
    ), row=1, col=idx+1)

    # 80% CI
    fig.add_trace(go.Scatter(
        x=val_dates + val_dates[::-1],
        y=list(np.clip(fc_val_df["yhat_upper"].values, 0, None)) +
          list(np.clip(fc_val_df["yhat_lower"].values, 0, None))[::-1],
        fill="toself",
        fillcolor=f"rgba(37,99,235,0.12)" if idx==0 else "rgba(217,119,6,0.12)",
        line=dict(width=0), name="80% CI", showlegend=(idx==0)
    ), row=1, col=idx+1)

    # baseline
    bl_vals = (
        df_val
        .with_columns(pl.col("dep_date").dt.month().alias("month"))
        .join(monthly_means_legs, on="month")
        .select(f"{col}_baseline").to_series().to_list()
    )
    fig.add_trace(go.Scatter(
        x=val_dates, y=bl_vals,
        mode="lines", name="Baseline",
        line=dict(color="#9ca3af", width=1.5, dash="dash")
    ), row=1, col=idx+1)

fig.update_layout(
    title="Leg Count Models — Prophet vs Baseline vs Actuals (Oct–Dec 2025)",
    height=600, width=1200,
    margin=dict(l=50, r=50, t=80, b=80)
)
fig.show()

In [5]:
import pickle

jan_may_dates = pd.date_range("2026-01-01", "2026-05-31", freq="D")

# slice Jan-May 2026 from already-computed full forecasts
fc_pred_prog = fc_prog_full[
    fc_prog_full["ds"].isin(jan_may_dates)
].reset_index(drop=True)

fc_pred_nonprog = fc_nonprog_full[
    fc_nonprog_full["ds"].isin(jan_may_dates)
].reset_index(drop=True)

prog_pred    = np.clip(fc_pred_prog["yhat"].values, 0, None).round(2)
nonprog_pred = np.clip(fc_pred_nonprog["yhat"].values, 0, None).round(2)

# build Jan-May 2026 legs forecast df
df_pred_legs = pl.DataFrame({
    "date"                    : list(jan_may_dates),
    "program_live_legs_fc"    : prog_pred.tolist(),
    "non_program_live_legs_fc": nonprog_pred.tolist(),
})

print("── Jan-May 2026 Leg Forecasts ──")
print(df_pred_legs)
print(f"\nMean program live legs (Jan-May 2026):      {prog_pred.mean():.2f}")
print(f"Mean non-program live legs (Jan-May 2026):  {nonprog_pred.mean():.2f}")

# save
df_pred_legs.write_parquet(PROCESSED_DIR / "jan2026_may2026_legs_forecast.parquet")

with open(MODELS_DIR / "legs_models.pkl", "wb") as f:
    pickle.dump({
        "program_prophet"     : model_prog,
        "non_program_prophet" : model_nonprog,
        "cancel_lookup"       : cancel_lookup,
        "baseline_metrics"    : results_baseline,
        "train_end"           : TRAIN_END,
    }, f)

print("\n✅ Saved → data/processed/jan2026_may2026_legs_forecast.parquet")
print("✅ Saved → models/legs_models.pkl")

── Jan-May 2026 Leg Forecasts ──
shape: (151, 3)
┌─────────────────────┬──────────────────────┬──────────────────────────┐
│ date                ┆ program_live_legs_fc ┆ non_program_live_legs_fc │
│ ---                 ┆ ---                  ┆ ---                      │
│ datetime[μs]        ┆ f64                  ┆ f64                      │
╞═════════════════════╪══════════════════════╪══════════════════════════╡
│ 2026-01-01 00:00:00 ┆ 26.86                ┆ 5.89                     │
│ 2026-01-02 00:00:00 ┆ 26.56                ┆ 6.53                     │
│ 2026-01-03 00:00:00 ┆ 17.96                ┆ 8.9                      │
│ 2026-01-04 00:00:00 ┆ 27.82                ┆ 6.0                      │
│ 2026-01-05 00:00:00 ┆ 24.39                ┆ 6.98                     │
│ …                   ┆ …                    ┆ …                        │
│ 2026-05-27 00:00:00 ┆ 23.76                ┆ 6.03                     │
│ 2026-05-28 00:00:00 ┆ 26.41                ┆ 5.33            